In [4]:
import pandas as pd
import numpy as np
import re
import spacy
from tqdm import tqdm

In [5]:
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded")

spaCy loaded


In [6]:
go = pd.read_csv("../data/raw/goemotions.csv")
reddit = pd.read_csv("../data/raw/reddit_mental_health.csv")
emotion = pd.read_csv("../data/raw/emotion.csv")

print(go.shape)
print(reddit.shape)
print(emotion.shape)

(54263, 3)
(16084, 1)
(16000, 2)


In [7]:
go = go[["text"]]
reddit = reddit[["text"]]
emotion = emotion[["text"]]

combined = pd.concat([go, reddit, emotion], ignore_index=True)

print("Combined shape:", combined.shape)
combined.head()

Combined shape: (86347, 1)


,text
0,My favourite food is anything I didn't have to...
1,"Now if he does off himself, everyone will thin..."
2,WHY THE FUCK IS BAYLESS ISOING
3,To make her feel threatened
4,Dirty Southern Wankers


In [8]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
tqdm.pandas()

combined["clean_text"] = combined["text"].progress_apply(clean_text)

combined.head()


100%|██████████| 86347/86347 [00:00<00:00, 124451.42it/s]


,text,clean_text
0,My favourite food is anything I didn't have to...,my favourite food is anything i didn t have to...
1,"Now if he does off himself, everyone will thin...",now if he does off himself everyone will think...
2,WHY THE FUCK IS BAYLESS ISOING,why the fuck is bayless isoing
3,To make her feel threatened,to make her feel threatened
4,Dirty Southern Wankers,dirty southern wankers


In [10]:
combined = combined[combined["clean_text"].str.len() > 5]
combined = combined.drop_duplicates(subset=["clean_text"])

print(combined.shape)

(85173, 2)


In [11]:
def preprocess_spacy(text):
    doc = nlp(text)

    tokens = []

    for token in doc:
        if token.is_stop:
            continue

        if token.is_punct:
            continue

        if token.is_space:
            continue

        lemma = token.lemma_.strip()

        if len(lemma) < 3:
            continue

        tokens.append(lemma)

    return " ".join(tokens)

In [12]:
combined["processed_text"] = combined["clean_text"].progress_apply(preprocess_spacy)

100%|██████████| 85173/85173 [09:01<00:00, 157.23it/s]


In [13]:
combined = combined[combined["processed_text"].str.len() > 5]

print(combined.shape)
combined.head()

(82872, 3)


,text,clean_text,processed_text
0,My favourite food is anything I didn't have to...,my favourite food is anything i didn t have to...,favourite food didn cook
1,"Now if he does off himself, everyone will thin...",now if he does off himself everyone will think...,think have laugh screw people instead actually...
2,WHY THE FUCK IS BAYLESS ISOING,why the fuck is bayless isoing,fuck bayless isoing
3,To make her feel threatened,to make her feel threatened,feel threaten
4,Dirty Southern Wankers,dirty southern wankers,dirty southern wanker


In [14]:
combined.to_csv("../data/processed/final_corpus.csv", index=False)

print("saved")

OSError: Cannot save file into a non-existent directory: '../data/processed'

In [15]:
import os

os.makedirs("../data/processed", exist_ok=True)

combined.to_csv("../data/processed/final_corpus.csv", index=False)

print("saved")

saved


In [16]:
combined[["text", "processed_text"]].sample(5)

,text,processed_text
49417,I hope you get the help you need. Truly.,hope help need truly
36158,He took it too far,take far
2349,Taking this way farther than needed...,take way far need
4122,Then the argument is entirely wrong. Democrati...,argument entirely wrong democratic socialist c...
57120,I've been trying to maintain my composure in t...,try maintain composure face disagreement ackno...
